# Package Installation

# Sudoku Solver

In [ ]:
from pysat.formula import CNF


def get_var(r, c, num, n):
    '''Map (row, column, number) to a unique variable index'''
    return r * n**2 + c * n + num


def get_coords(var_index, n):
    '''Map the variable index back to (row, column, number)'''
    num = (var_index - 1) % n + 1
    c = ((var_index - 1) // n) % n
    r = (var_index - 1) // (n**2)
    return r, c, num


def sudoku2sat(grid, k):
    """Reduce a Sudoku puzzle to a SAT problem.

    Args:
        grid (list[list[int]]): A square matrix of size k² × k².

    Returns:
        A CNF formula that is satisfiable if and only if
        the unspecified elements (i.e., entries not in [1, k])
        can be filled in to form a valid Sudoku solution.
    """
    n = len(grid)
    if n != k**2 or any(len(row) != n for row in grid):
        raise ValueError("Invalid Sudoku grid")

    f = CNF()

    # Map (row, column, number) to a unique variable index
    def var(r, c, num):
        return get_var(r, c, num, n)

    # Add clauses for initial grid values
    for r in range(n):
        for c in range(n):
            if grid[r][c] in range(1, n + 1):
                f.append([var(r, c, grid[r][c])])

    # Each cell must contain exactly one number
    for r in range(n):
        for c in range(n):
            # At least one number
            f.append([var(r, c, num) for num in range(1, n + 1)])
            # At most one number
            for num1 in range(1, n + 1):
                for num2 in range(num1 + 1, n + 1):
                    f.append([-var(r, c, num1), -var(r, c, num2)])

    # Each row must contain each number exactly once
    for r in range(n):
        for num in range(1, n + 1):
            f.append([var(r, c, num) for c in range(n)])
            for c1 in range(n):
                for c2 in range(c1 + 1, n):
                    f.append([-var(r, c1, num), -var(r, c2, num)])

    # Each column must contain each number exactly once
    for c in range(n):
        for num in range(1, n + 1):
            f.append([var(r, c, num) for r in range(n)])
            for r1 in range(n):
                for r2 in range(r1 + 1, n):
                    f.append([-var(r1, c, num), -var(r2, c, num)])

    # Each k x k subgrid must contain each number exactly once
    for sr in range(k):
        for sc in range(k):
            for num in range(1, n + 1):
                cells = [(r, c) for r in range(sr * k, (sr + 1) * k)
                                for c in range(sc * k, (sc + 1) * k)]
                f.append([var(r, c, num) for (r, c) in cells])
                for i in range(len(cells)):
                    for j in range(i + 1, len(cells)):
                        r1, c1 = cells[i]
                        r2, c2 = cells[j]
                        f.append([-var(r1, c1, num), -var(r2, c2, num)])

    return f


def assemble_solution(model, k):
    """Extract a Sudoku solution from a satisfying assignment of the formula
    obtained via reduction.

    Args:
        model: None if the formula is unsatisfiable;
               otherwise, a list specifying values of the variables in the
               satisfying assignment.

    Returns:
        A grid representing the Sudoku solution.
    """
    if model is None:
        return None

    n = k**2
    solution = [[0 for _ in range(n)] for _ in range(n)]

    for var_index in model:
        if var_index > 0:
            r, c, num = get_coords(var_index, n)
            solution[r][c] = num

    return solution

In [2]:
from pysat.solvers import Solver


def solve_sudoku(grid, k):
    """Use a SAT solver to solve the Sudoku puzzle"""

    print("Reducing to SAT...")
    f = sudoku2sat(grid, k)
    # print(f.clauses)

    print("Solving SAT...")
    with Solver(bootstrap_with=f.clauses) as s:
        s.solve()
        model = s.get_model()

    if (model):
        print("Assembling solution...")
        return assemble_solution(model, k)

    print("No solution found")
    return None


def print_solution(solution):
    if solution:
        for row in solution:
            print(" ".join(map(str, row)))

## Examples

In [3]:
# Example Sudoku grid (0 represents empty cells)
sudoku_grid = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

solve_sudoku(sudoku_grid, 3)

Reducing to SAT...
Solving SAT...
Assembling solution...


[[5, 3, 4, 6, 7, 8, 9, 1, 2],
 [6, 7, 2, 1, 9, 5, 3, 4, 8],
 [1, 9, 8, 3, 4, 2, 5, 6, 7],
 [8, 5, 9, 7, 6, 1, 4, 2, 3],
 [4, 2, 6, 8, 5, 3, 7, 9, 1],
 [7, 1, 3, 9, 2, 4, 8, 5, 6],
 [9, 6, 1, 5, 3, 7, 2, 8, 4],
 [2, 8, 7, 4, 1, 9, 6, 3, 5],
 [3, 4, 5, 2, 8, 6, 1, 7, 9]]

In [5]:
sudoku_grid2 = [
    [4, 3, 0, 0],
    [2, 0, 0, 1],
    [0, 4, 3, 0],
    [3, 0, 0, 0]
]

solve_sudoku(sudoku_grid2, 2)

Reducing to SAT...
Solving SAT...
No solution found


In [6]:
sudoku_grid2 = [
    [4, 3, 0, 0],
    [2, 0, 0, 3],
    [0, 4, 3, 0],
    [3, 0, 0, 0]
]

print_solution(solve_sudoku(sudoku_grid2, 2))

Reducing to SAT...
Solving SAT...
Assembling solution...
4 3 2 1
2 1 4 3
1 4 3 2
3 2 1 4


In [7]:
sudoku_grid3 = [
    [1, 0, 0, 0, 0, 0, 0, 0, 0],
    [2, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 4, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0]
]

solve_sudoku(sudoku_grid3, 3)

Reducing to SAT...
Solving SAT...
Assembling solution...


[[1, 6, 8, 2, 3, 9, 7, 4, 5],
 [2, 4, 5, 8, 7, 6, 1, 3, 9],
 [7, 9, 3, 4, 1, 5, 2, 6, 8],
 [4, 3, 6, 1, 8, 7, 5, 9, 2],
 [5, 2, 1, 9, 4, 3, 8, 7, 6],
 [8, 7, 9, 5, 6, 2, 4, 1, 3],
 [6, 5, 4, 3, 2, 1, 9, 8, 7],
 [3, 1, 2, 7, 9, 8, 6, 5, 4],
 [9, 8, 7, 6, 5, 4, 3, 2, 1]]

In [5]:
sudoku_grid4 = [
    [2, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0]
]

print_solution(solve_sudoku(sudoku_grid4, 2))

Reducing to SAT...
Solving SAT...
Assembling solution...
2 1 3 4
3 4 1 2
1 2 4 3
4 3 2 1


# Exploration without Negations

In [3]:
from core.partial_object import PartialObject
from exploration.attribute_exploration import AttributeExploration
from experts.expert import Expert
from exploration.exploration_base import ExplorationBase


class ANotBExpert(Expert):
    def validate(self, impl, attributes=None):
        return (PartialObject('a', {'a'}, {'b'})
                if impl.premise <= {'a'} and 'b' in impl.conclusion
                else None)


base = ExplorationBase(attributes=['a', 'b'])
exploration = AttributeExploration(base, ANotBExpert())
exploration.run()
for impl in base.implications:
    print(impl)

 -> a


## Sudoku Expert

In [4]:
class SudokuExpert(Expert):

    def __init__(self, block_size):
        self.block_size = block_size

    def validate(self, implication, attributes=None):
        n = self.block_size**2

        grid = [[0]*n for _ in range(n)]
        for (r, c, num) in implication.premise:
            if grid[r][c] == 0:
                grid[r][c] = num
            else:
                return None

        f = sudoku2sat(grid, self.block_size)
        f.append([-get_var(r, c, num, n) for (r, c, num) in implication.conclusion])

        with Solver(bootstrap_with=f.clauses) as s:
            s.solve()
            model = s.get_model()

        if model:
            return PartialObject(
                str(implication),
                set(get_coords(a, n) for a in model if a > 0),
                set(get_coords(abs(a), n) for a in model if a < 0)
            )

        return None


In [5]:
from conceptual_exploration.core.implication import Implication

impl = Implication({(0, 0, 1), (1, 0, 2), (2, 3, 4)}, {(2, 0, 3), (3, 0, 4)})
SudokuExpert(3).validate(impl).negative


{(0, 0, 2),
 (0, 0, 3),
 (0, 0, 4),
 (0, 0, 5),
 (0, 0, 6),
 (0, 0, 7),
 (0, 0, 8),
 (0, 0, 9),
 (0, 1, 1),
 (0, 1, 2),
 (0, 1, 3),
 (0, 1, 4),
 (0, 1, 5),
 (0, 1, 7),
 (0, 1, 8),
 (0, 1, 9),
 (0, 2, 1),
 (0, 2, 2),
 (0, 2, 3),
 (0, 2, 4),
 (0, 2, 5),
 (0, 2, 6),
 (0, 2, 7),
 (0, 2, 9),
 (0, 3, 1),
 (0, 3, 3),
 (0, 3, 4),
 (0, 3, 5),
 (0, 3, 6),
 (0, 3, 7),
 (0, 3, 8),
 (0, 3, 9),
 (0, 4, 1),
 (0, 4, 2),
 (0, 4, 4),
 (0, 4, 5),
 (0, 4, 6),
 (0, 4, 7),
 (0, 4, 8),
 (0, 4, 9),
 (0, 5, 1),
 (0, 5, 2),
 (0, 5, 3),
 (0, 5, 4),
 (0, 5, 5),
 (0, 5, 6),
 (0, 5, 7),
 (0, 5, 8),
 (0, 6, 1),
 (0, 6, 2),
 (0, 6, 3),
 (0, 6, 4),
 (0, 6, 5),
 (0, 6, 6),
 (0, 6, 8),
 (0, 6, 9),
 (0, 7, 1),
 (0, 7, 2),
 (0, 7, 3),
 (0, 7, 5),
 (0, 7, 6),
 (0, 7, 7),
 (0, 7, 8),
 (0, 7, 9),
 (0, 8, 1),
 (0, 8, 2),
 (0, 8, 3),
 (0, 8, 4),
 (0, 8, 6),
 (0, 8, 7),
 (0, 8, 8),
 (0, 8, 9),
 (1, 0, 1),
 (1, 0, 3),
 (1, 0, 4),
 (1, 0, 5),
 (1, 0, 6),
 (1, 0, 7),
 (1, 0, 8),
 (1, 0, 9),
 (1, 1, 1),
 (1, 1, 2),
 (1, 1, 3),
 (1,

## Sudoku Exploration

## Attributes

In [5]:
k = 2
n = k**2
attributes=[
    (r, c, num)
    for r in range(n)
    for c in range(n)
    for num in range(1, n + 1)
]

## Background

In [7]:
from itertools import combinations, permutations

def row_cells(r, c):
    return [(r, y) for y in range(n) if y != c]


def column_cells(r, c):
    return [(x, c) for x in range(n) if x != r]


def block_cells(r, c, k):
    return [
        (x, y)
        for x in range(r//k * k, (r//k + 1) * k)
        for y in range(c//k * k, (c//k + 1) * k)
        if x != r or y != c
    ]


def n_1_implications(cells, r, c, num):
    for p in permutations(cells):
        yield Implication(
            {
                (r1, c1, num1)
                for (r1, c1), num1 in zip(p, numbers)
            },
            {(r, c, num)}
        )


background = [
    Implication({(r1, c1, n1), (r2, c2, n2)}, attributes)
    for ((r1, c1, n1), (r2, c2, n2)) in combinations(attributes, 2)
    if (r1 == r2 and c1 == c2)  # at most one digit per cell
       or (n1 == n2 # all digits within a row/column/block differ
           and (r1 == r2 or c1 == c2 or (r1//k == r2//k and c1//k == c2//k))
           )
]
print(len(background))

for (r, c, num) in attributes:
    numbers = [m for m in range(1, n + 1) if m != num]
    for i in n_1_implications(row_cells(r, c), r, c, num):
        background.append(i)    # last digit in a row
    for i in n_1_implications(column_cells(r, c), r, c, num):
        background.append(i)    # last digit in a column
    for i in n_1_implications(block_cells(r, c, k), r, c, num):
        background.append(i)    # last digit in a block
print(len(background))

320
1472


## Symmetries

In [6]:
from itertools import permutations


def make_number_mapping(permutation):
    return lambda r_c_num: (
        r_c_num[0],
        r_c_num[1],
        permutation[r_c_num[2] - 1]
    )


numbers = tuple(range(1, n + 1))
number_mappings = [
    make_number_mapping(permutation)
    for permutation in permutations(numbers)
    if permutation != numbers
]
print(len(number_mappings))

rotation_mappings = [
    lambda r_c_num: (r_c_num[1], n - 1 - r_c_num[0], r_c_num[2]),
    lambda r_c_num: (n - 1 - r_c_num[0], n - 1 - r_c_num[1], r_c_num[2]),
    lambda r_c_num: (n - 1 - r_c_num[1], r_c_num[0], r_c_num[2])
]

compositions = [
    lambda r_c_num: rotation(number(r_c_num))
    for rotation in rotation_mappings
    for number in number_mappings
]
print(len(compositions))

mirror_mappings = [
    lambda r_c_num: (n - 1 - r_c_num[0], r_c_num[1], r_c_num[2]),
    lambda r_c_num: (r_c_num[1], n - 1 - r_c_num[0], r_c_num[2])
]

23
69


In [22]:
impl = Implication({(2, 3, 4), (3, 1, 3)}, {(2, 2, 3), (3, 0, 4)})

for mapping in number_mappings:
    mapped_implication = Implication(
        frozenset(mapping(a) for a in impl.premise),
        frozenset(mapping(a) for a in impl.conclusion)
    )
    print(mapped_implication)


(2, 3, 3), (3, 1, 4) -> (2, 2, 4), (3, 0, 3)
(2, 3, 4), (3, 1, 2) -> (2, 2, 2), (3, 0, 4)
(2, 3, 2), (3, 1, 4) -> (2, 2, 4), (3, 0, 2)
(2, 3, 3), (3, 1, 2) -> (2, 2, 2), (3, 0, 3)
(2, 3, 2), (3, 1, 3) -> (2, 2, 3), (3, 0, 2)
(2, 3, 4), (3, 1, 3) -> (2, 2, 3), (3, 0, 4)
(2, 3, 3), (3, 1, 4) -> (2, 2, 4), (3, 0, 3)
(2, 3, 4), (3, 1, 1) -> (2, 2, 1), (3, 0, 4)
(2, 3, 1), (3, 1, 4) -> (2, 2, 4), (3, 0, 1)
(2, 3, 3), (3, 1, 1) -> (2, 2, 1), (3, 0, 3)
(2, 3, 1), (3, 1, 3) -> (2, 2, 3), (3, 0, 1)
(2, 3, 4), (3, 1, 2) -> (2, 2, 2), (3, 0, 4)
(2, 3, 2), (3, 1, 4) -> (2, 2, 4), (3, 0, 2)
(2, 3, 4), (3, 1, 1) -> (2, 2, 1), (3, 0, 4)
(2, 3, 1), (3, 1, 4) -> (2, 2, 4), (3, 0, 1)
(2, 3, 2), (3, 1, 1) -> (2, 2, 1), (3, 0, 2)
(2, 3, 1), (3, 1, 2) -> (2, 2, 2), (3, 0, 1)
(2, 3, 3), (3, 1, 2) -> (2, 2, 2), (3, 0, 3)
(2, 3, 2), (3, 1, 3) -> (2, 2, 3), (3, 0, 2)
(2, 3, 3), (3, 1, 1) -> (2, 2, 1), (3, 0, 3)
(2, 3, 1), (3, 1, 3) -> (2, 2, 3), (3, 0, 1)
(2, 3, 2), (3, 1, 1) -> (2, 2, 1), (3, 0, 2)
(2, 3, 1),

In [23]:
impl = Implication({(2, 1, 3), (3, 3, 4)}, {(2, 2, 3), (3, 0, 4)})

for mapping in rotation_mappings:
    mapped_implication = Implication(
        frozenset(mapping(a) for a in impl.premise),
        frozenset(mapping(a) for a in impl.conclusion)
    )
    print(mapped_implication)

(1, 1, 3), (3, 0, 4) -> (0, 0, 4), (2, 1, 3)
(0, 0, 4), (1, 2, 3) -> (0, 3, 4), (1, 1, 3)
(0, 3, 4), (2, 2, 3) -> (1, 2, 3), (3, 3, 4)


## Exploration

In [14]:
base = ExplorationBase(
    attributes=attributes,
    #background_implications=background,
    mappings=number_mappings+rotation_mappings#+mirror_mappings
    #mappings=rotation_mappings
)
exploration = AttributeExploration(base, SudokuExpert(k))

In [15]:
len(attributes), len(base.mappings)

(64, 26)

In [16]:
res = exploration.run()


Number of implications: 191
Number of objects: 108
Question 20: (2, 3, 4), (3, 2, 3), (3, 3, 2) -> (2, 2, 1)
Confirmed

Number of implications: 456
Number of objects: 189
Question 40: (2, 1, 4), (2, 3, 4) -> (0, 0, 1), (0, 0, 2), (0, 0, 3), (0, 0, 4), (0, 1, 1), (0, 1, 2), (0, 1, 3), (0, 1, 4), (0, 2, 1), (0, 2, 2), (0, 2, 3), (0, 2, 4), (0, 3, 1), (0, 3, 2), (0, 3, 3), (0, 3, 4), (1, 0, 1), (1, 0, 2), (1, 0, 3), (1, 0, 4), (1, 1, 1), (1, 1, 2), (1, 1, 3), (1, 1, 4), (1, 2, 1), (1, 2, 2), (1, 2, 3), (1, 2, 4), (1, 3, 1), (1, 3, 2), (1, 3, 3), (1, 3, 4), (2, 0, 1), (2, 0, 2), (2, 0, 3), (2, 0, 4), (2, 1, 1), (2, 1, 2), (2, 1, 3), (2, 2, 1), (2, 2, 2), (2, 2, 3), (2, 2, 4), (2, 3, 1), (2, 3, 2), (2, 3, 3), (3, 0, 1), (3, 0, 2), (3, 0, 3), (3, 0, 4), (3, 1, 1), (3, 1, 2), (3, 1, 3), (3, 1, 4), (3, 2, 1), (3, 2, 2), (3, 2, 3), (3, 2, 4), (3, 3, 1), (3, 3, 2), (3, 3, 3), (3, 3, 4)
Confirmed

Number of implications: 719
Number of objects: 270
Question 60: (2, 0, 3), (3, 0, 3) -> (0, 0, 1), 

In [17]:
len(exploration.base.implications.implications)

5056

In [18]:
len(exploration.state.accepted_implications)

312

In [19]:
len(exploration.state.counterexamples)

12

In [20]:
exploration.state.questions_asked

324

In [31]:
for impl in exploration.state.accepted_implications:
    print(impl)

(2, 3, 4), (3, 1, 3) -> (2, 2, 3), (3, 0, 4)
(2, 3, 4), (3, 0, 3) -> (2, 2, 3), (3, 1, 4)
(2, 2, 4), (3, 1, 3) -> (2, 3, 3), (3, 0, 4)
(2, 2, 4), (3, 0, 3) -> (2, 3, 3), (3, 1, 4)
(2, 1, 4), (3, 3, 3) -> (2, 0, 3), (3, 2, 4)
(2, 1, 4), (3, 2, 3) -> (2, 0, 3), (3, 3, 4)
(2, 0, 4), (3, 3, 3) -> (2, 1, 3), (3, 2, 4)
(2, 0, 4), (3, 2, 3) -> (2, 1, 3), (3, 3, 4)
(1, 3, 4), (3, 2, 3) -> (0, 3, 3), (2, 2, 4)
(1, 3, 4), (3, 1, 4) -> (0, 0, 4), (2, 2, 4)
(1, 3, 4), (3, 0, 4) -> (0, 1, 4), (2, 2, 4)
(1, 3, 4), (3, 0, 3), (3, 1, 2) -> (3, 2, 4), (3, 3, 1)
(1, 3, 4), (2, 2, 3) -> (0, 3, 3), (3, 2, 4)
(1, 3, 4), (2, 1, 4) -> (0, 0, 4), (3, 2, 4)
(1, 3, 4), (2, 1, 3), (3, 1, 2) -> (0, 1, 4), (1, 1, 1)
(1, 3, 4), (2, 0, 4) -> (0, 1, 4), (3, 2, 4)
(1, 3, 4), (2, 0, 3), (3, 0, 2) -> (0, 0, 4), (1, 0, 1)
(1, 3, 4), (2, 0, 3), (2, 1, 2) -> (2, 2, 4), (2, 3, 1)
(1, 2, 4), (3, 3, 2) -> (0, 2, 2), (2, 3, 4)
(1, 2, 4), (3, 1, 4) -> (0, 0, 4), (2, 3, 4)
(1, 2, 4), (3, 0, 4) -> (0, 1, 4), (2, 3, 4)
(1, 2, 4), 